# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


Rule
- Flag content for manual review when it has a measurable audience (impressions_90d) but low user engagement (low CTR) or an unexpectedly poor / missing search position (avg_position). Prioritise items with larger audiences first.

Why this rule?
- This is human-readable and targets items with potential impact: pages that many people see but few engage with, or pages whose search position is missing/poor and therefore deserve editorial attention.
- It avoids derived/label features (no trend_pct or trend_direction used)

Reason codes
- high_impr_low_ctr — many impressions, low CTR
- high_impr_poor_position — many impressions, but position is poor
- high_impr_missing_position — many impressions, but no position data
- moderate_impr_low_ctr — medium impressions + low CTR (lower confidence)
- default_review — fallback, small signals only

Confidence notes (used in top-20 review)
- high: multiple strong signals (e.g., high impressions AND low ctr)
- medium: one clear signal with moderate impressions
- low: borderline signals or tiny audiences

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
# Robust loader: try local file first, then download from GitHub raw URL if missing.
import os
import numpy as np
import pandas as pd

RAW_URL = "https://raw.githubusercontent.com/reezcon/First-ML-Pipeline/main/data/raw/content_refresh_anonymized.csv"
OUT_DIR = "work/outputs"
OUT_CSV = os.path.join(OUT_DIR, "baseline_action_score.csv")
os.makedirs(OUT_DIR, exist_ok=True)

# Helpful debug: where are we and what files exist at top level
print("Working directory:", os.getcwd())
print("Top-level files/dirs:", sorted([p for p in os.listdir(".") if not p.startswith(".")])[:50])

try:
    df = pd.read_csv(RAW_URL)
    print("Downloaded CSV from GitHub.")
except Exception as e:
    raise RuntimeError(
        "Failed to load CSV from local path and from GitHub raw URL. "
        "If you are offline or GitHub is blocked, please git-clone the repo or upload the CSV to data/raw/."
    ) from e

print("Loaded rows:", len(df), "columns:", len(df.columns))
print("Sample columns:", list(df.columns)[:20])

# Now continue with the simple baseline logic (same as before) ...
# Convert numeric columns and mark missing position
df["impressions_90d"] = pd.to_numeric(df.get("impressions_90d", df.get("impressions", 0)), errors="coerce").fillna(0)
if "ctr" in df.columns:
    df["ctr"] = pd.to_numeric(df["ctr"], errors="coerce")
if "avg_position" in df.columns:
    df["avg_position"] = pd.to_numeric(df["avg_position"], errors="coerce")
    df["avg_position_missing"] = ((df["avg_position"] == 0) | df["avg_position"].isna()).astype(int)
else:
    df["avg_position_missing"] = 1

# Thresholds and signals (simple and readable)
TH_IMPR_HIGH = 1000
TH_IMPR_MED = 500
TH_CTR_LOW = 0.5
TH_POSITION_POOR = 10

df["high_impr"] = (df["impressions_90d"] >= TH_IMPR_HIGH).astype(int)
df["med_impr"] = ((df["impressions_90d"] >= TH_IMPR_MED) & (df["impressions_90d"] < TH_IMPR_HIGH)).astype(int)
df["low_ctr"] = ((df.get("ctr").notna()) & (df["ctr"] <= TH_CTR_LOW)).astype(int) if "ctr" in df.columns else 0
df["poor_position"] = ((df.get("avg_position").notna()) & (df["avg_position"] > TH_POSITION_POOR) & (df["avg_position_missing"] == 0)).astype(int)
df["missing_position"] = df["avg_position_missing"].astype(int)

df["score"] = 3*df["high_impr"] + 2*df["med_impr"] + 2*df["low_ctr"] + 2*df["poor_position"] + 1*df["missing_position"]

def make_reason(r):
    if r["high_impr"] and r["low_ctr"]:
        return "high_impr_low_ctr"
    if r["high_impr"] and r["poor_position"]:
        return "high_impr_poor_position"
    if r["high_impr"] and r["missing_position"]:
        return "high_impr_missing_position"
    if r["med_impr"] and r["low_ctr"]:
        return "moderate_impr_low_ctr"
    return "default_review"

df["reason_code"] = df.apply(make_reason, axis=1)

# Rank and save output CSV
df = df.sort_values(["score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
df["rank"] = np.arange(1, len(df)+1)

out_cols = []
for c in ("content_id","client_id"):
    if c in df.columns:
        out_cols.append(c)
out_cols += ["score","rank","reason_code","impressions_90d"]
if "ctr" in df.columns:
    out_cols.append("ctr")
if "avg_position" in df.columns:
    out_cols += ["avg_position","avg_position_missing"]
if "engagement_rate" in df.columns:
    out_cols.append("engagement_rate")

df[out_cols].to_csv(OUT_CSV, index=False)
print("Wrote", OUT_CSV, "rows:", len(df))

Working directory: /content
Top-level files/dirs: ['sample_data', 'work']
Downloaded CSV from GitHub.
Loaded rows: 30000 columns: 44
Sample columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d']
Wrote work/outputs/baseline_action_score.csv rows: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [2]:
# Evaluation: base rate + precision@K if engagement_rate exists
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = order[:k]
    return np.asarray(labels).astype(float)[topk].mean()

if "engagement_rate" in df.columns:
    y = df["engagement_rate"].fillna(0).astype(float)
    base = y.mean()
    print(f"Base rate (engagement_rate mean) = {base:.4f}")
    for k in (20,50,100):
        print(f"Precision@{k} =", f"{precision_at_k(df['score'].values, y, k):.4f}")
else:
    print("No engagement_rate label found; skipping precision@K.")

Base rate (engagement_rate mean) = 2.5345
Precision@20 = 1.1805
Precision@50 = 1.4540
Precision@100 = 1.5476


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.